In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time

# --- 1. System Setup ---
bln69_sequence = "BBBBBBBBBNNNLBLBLBLBNNNBBBBBBBBBNNNLBLBLBLBNNNBBBBBBBBBNNNLBLBLBLBLBL"
protein_types = np.array([{'B': 0, 'L': 1, 'N': 2}[res] for res in bln69_sequence])
n_residues = len(bln69_sequence)

initial_coords = np.zeros((n_residues, 3))
initial_coords[:, 0] = np.arange(n_residues)

# --- 2. Shared Physics & Geometry Helpers ---
def get_rotation_matrix(axis, theta):
    axis = axis / np.linalg.norm(axis)
    a = np.cos(theta / 2.0)
    b, c, d = -axis * np.sin(theta / 2.0)
    aa, bb, cc, dd = a*a, b*b, c*c, d*d
    bc, ad, ac, ab, bd, cd = b*c, a*d, a*c, a*b, b*d, c*d
    return np.array([
        [aa+bb-cc-dd, 2*(bc+ad), 2*(bd-ac)],
        [2*(bc-ad), aa+cc-bb-dd, 2*(cd+ab)],
        [2*(bd+ac), 2*(cd-ab), aa+dd-bb-cc]
    ])

def bln_potential(coords, types):
    # Calculate distances and enforce 0.1 lower bound concisely
    dist = np.linalg.norm(coords[:, np.newaxis, :] - coords[np.newaxis, :, :], axis=2)
    dist = np.maximum(dist, 0.1) 
    
    types_grid_i, types_grid_j = np.meshgrid(types, types, indexing='ij')
    bb_mask = (types_grid_i == 0) & (types_grid_j == 0)
    
    inv_r6 = (1.0 / dist)**6
    inv_r12 = inv_r6**2
    
    # Calculate energies for all pairs cleanly in one matrix
    energy_matrix = np.where(bb_mask, 4.0 * (inv_r12 - inv_r6), 0.5 * inv_r12)
    mask_upper = np.triu(np.ones_like(energy_matrix, dtype=bool), k=2)
    
    return np.sum(energy_matrix[mask_upper])

def propose_move(coords, max_angle=0.5):
    n = len(coords)
    idx = np.random.randint(1, n - 1)
    axis = np.random.randn(3)
    angle = (np.random.rand() - 0.5) * 2 * max_angle
    
    candidate_coords = np.copy(coords)
    pivot_point = candidate_coords[idx]
    moving_segment = candidate_coords[idx + 1:] - pivot_point
    rot_matrix = get_rotation_matrix(axis, angle)
    candidate_coords[idx + 1:] = np.dot(moving_segment, rot_matrix.T) + pivot_point
    
    return candidate_coords, abs(angle)

def evaluate_acceptance(delta_E, T, L, mode, alpha=3.64):
    if delta_E <= 0:
        return True
    if T < 1e-10:
        return False
        
    if mode == 'classical':
        if delta_E / T < 50:
            return np.random.rand() < np.exp(-delta_E / T)
        return False
        
    elif mode == 'quantum':
        kappa_L = alpha * L * np.sqrt(delta_E)
        if kappa_L < 50:
            v0 = T + delta_E
            prefactor = (v0**2) / (4 * T * delta_E)
            try:
                p_tunnel = 1.0 / (1.0 + prefactor * (np.sinh(kappa_L)**2))
                return np.random.rand() < p_tunnel
            except OverflowError:
                return False
        return False
    
    raise ValueError("Mode must be 'classical' or 'quantum'")

def run_simulation(coords_init, types, steps, T_start=50.0, cooling_rate=0.995, mode='quantum', alpha=2.93):
    coords = np.copy(coords_init)
    best_coords = np.copy(coords)
    current_energy = bln_potential(coords, types)
    best_energy = current_energy
    
    history = []
    T = T_start
    
    for _ in range(steps):
        candidate_coords, L = propose_move(coords)
        
        candidate_energy = bln_potential(candidate_coords, types)
        delta_E = candidate_energy - current_energy
        
        if evaluate_acceptance(delta_E, T, L, mode, alpha):
            coords = candidate_coords
            current_energy = candidate_energy
            if current_energy < best_energy:
                best_energy = current_energy
                best_coords = np.copy(coords)
                
        history.append(current_energy)
        T *= cooling_rate
        
    return best_coords, best_energy, history